In [68]:
# !pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-proto -q
# !pip install chromadb==1.5.9 sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-exporter-gcp-logging<2.0.0,>=1.9.0a0, which is not installed.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, which is not installed.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.


In [69]:
# import chromadb
import sentence_transformers
import pandas as pd
import numpy as np

# print(f"ChromaDB version: {chromadb.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Sentence Transformers version: 5.5.1
Pandas version: 2.2.2
Numpy version: 2.0.2


In [70]:
documents=[
    "ETL is used to clean and transfrom data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobile",
    "SQL is used to query databases",
    "Machine learning train models in data",
]

query_keyword="Vehicle"
print("="*50)
print("Keyword search for:",query_keyword)
print("="*50)

for i,doc in enumerate(documents):
    if query_keyword.lower() in doc.lower():
      print("Found",doc)
    else:
      print("Not found",doc)
print()
print("Problem doc_2 talks about cars and trucks which are vehicles ")
print("but keyword search missed it became it searched for the exact word")

Keyword search for: Vehicle
Not found ETL is used to clean and transfrom data
Found A vehicle is a mode of transportation
Not found Cars and trucks are popular automobile
Not found SQL is used to query databases
Not found Machine learning train models in data

Problem doc_2 talks about cars and trucks which are vehicles 
but keyword search missed it became it searched for the exact word


In [71]:
failure_examples=[
    {"query":"I  feel sick" ,  "misses":" I am unwell,patient has fever"},
    {"query":"How to cook rice"  , "misses":"Steps to prepare rice"},
    {"query":"Vehicle speed"  , "misses":"Car accleration and automobile vehicle"},
    {"query":"ML model accuracy" , "misses":"Classification ,performance,prediction quality"},
]

print("Keyword  search failure cases:")
print("="*30)
for ex in failure_examples:
  print("Query:",ex['query'])
  print("Misses:",ex['misses'])
  print("="*30)
print()
print("Solution: we need to search that meaning and understand,not a character")
print("That is what embeddings do!!")

Keyword  search failure cases:
Query: I  feel sick
Misses:  I am unwell,patient has fever
Query: How to cook rice
Misses: Steps to prepare rice
Query: Vehicle speed
Misses: Car accleration and automobile vehicle
Query: ML model accuracy
Misses: Classification ,performance,prediction quality

Solution: we need to search that meaning and understand,not a character
That is what embeddings do!!


In [72]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model....")
model=SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully")

Loading embedding model....


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [73]:
sentence="ETL is used to clean and tansform data"
embedding=model.encode(sentence)
print()
print("Input sentence",sentence)
print("Embedding type:",type(embedding) )
print("Embedding shape:",embedding.shape)
print("First 10 numbers:",embedding[:10])


Input sentence ETL is used to clean and tansform data
Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 numbers: [-0.058379    0.03554146  0.01654611 -0.02895406  0.03935834 -0.0858812
  0.03790451 -0.00531707  0.0565182   0.02839596]


In [74]:
sentences=[
    "ETL is used to clean and transfrom data",
    "Data transformation is a key pipeline step",
    "The sky is blue and clouds are white",
]
embeddings=model.encode(sentences)
print("Number of sentences:",len(sentences))
print("Embeddings type:",type(embeddings))
print("Embeddings shape:",embeddings.shape)

print()
print("Each row is one sentence''s embedding")
for i , sent in enumerate(sentences):
  print("Sentence:",embeddings[i][:5].round(3))

Number of sentences: 3
Embeddings type: <class 'numpy.ndarray'>
Embeddings shape: (3, 384)

Each row is one sentence''s embedding
Sentence: [-0.06   0.029  0.024 -0.029  0.022]
Sentence: [-0.047  0.059 -0.001 -0.033 -0.046]
Sentence: [0.054 0.06  0.075 0.049 0.05 ]


In [75]:
def cosine_similarity(vec_a,vec_b):
  dot_pro=np.dot(vec_a,vec_b)
  norm_a=np.linalg.norm(vec_a)
  norm_b=np.linalg.norm(vec_b)
  return dot_pro/(norm_a*norm_b)
sin_01=cosine_similarity(embeddings[0],embeddings[1])
sin_02=cosine_similarity(embeddings[0],embeddings[2])
sin_12=cosine_similarity(embeddings[1],embeddings[2])
print("Cosine similarity scores")
print("="*60)
print("Sentence 0:",sentences[0])
print("Sentence 1:",sentences[1])
print("Sentence 2:",sentences[2])
print()
print(f"Similarity(0 vs 1):{sin_01:.4f}")
print(f"Similarity(0 vs 2):{sin_02:.4f}")
print(f"Similarity(1vs 2):{sin_12:.4f}")
print()
print("INSIGHT: Sentence 0 and 1 have different meaning")
print("Their cosine similarity score is  high-embedding captured the meaning")

Cosine similarity scores
Sentence 0: ETL is used to clean and transfrom data
Sentence 1: Data transformation is a key pipeline step
Sentence 2: The sky is blue and clouds are white

Similarity(0 vs 1):0.3593
Similarity(0 vs 2):-0.0337
Similarity(1vs 2):0.0363

INSIGHT: Sentence 0 and 1 have different meaning
Their cosine similarity score is  high-embedding captured the meaning


In [76]:
your_sentences=[
    "Machine Learning trains models",
    "I love  cats and dogs",
    "My goal is to become a cloud engineer and data engineer",
    "Supervised learninng and unsupervised learning",
]
your_embeddings=model.encode(your_sentences)
sin_your_01=cosine_similarity(your_embeddings[0],your_embeddings[1])
sin_your_02=cosine_similarity(your_embeddings[0],your_embeddings[2])
sin_your_03=cosine_similarity(your_embeddings[0],your_embeddings[3])
sin_your_12=cosine_similarity(your_embeddings[1],your_embeddings[2])

print("YOUR EXPERIMENT RESULTS")
print("="*60)
print("Sentence 0:",your_sentences[0])
print("Sentence 1:",your_sentences[1])
print("Sentence 2:",your_sentences[2])
print("Sentence 3:",your_sentences[3])
print()
print(f"Similarity(0 vs 1):{sin_your_01:.4f}")
print(f"Similarity(0 vs 2):{sin_your_02:.4f}")
print(f"Similarity(0 vs 3):{sin_your_03:.4f}")
print(f"Similarity(1vs 2):{sin_your_12:.4f}")




YOUR EXPERIMENT RESULTS
Sentence 0: Machine Learning trains models
Sentence 1: I love  cats and dogs
Sentence 2: My goal is to become a cloud engineer and data engineer
Sentence 3: Supervised learninng and unsupervised learning

Similarity(0 vs 1):0.0000
Similarity(0 vs 2):0.0026
Similarity(0 vs 3):0.3983
Similarity(1vs 2):0.1344


In [77]:
!pip install -U chromadb

In [82]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.9 MB/s eta 0:00:00


In [3]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("demo")
print("ChromaDB client created successfully")
print("Collection name:Demo")
print("Documents in collection(demo):",collection.count)


ChromaDB client created successfully
Collection name:Demo
Documents in collection(demo): <bound method Collection.count of Collection(name=demo)>


In [4]:
#Let us add 5 sample documents
sample_docs=[
    "ETL stands for Extract Transform Load-the core data engineering process",
    "SQL select statements retrieve data from database tables",
    "Machine Learning models learn pattern from training data",
    "Python Pandas Library is used for data manipulation and cleaning",
    "Neural Networks are inspired by how the human brain works"
]
sample_ids=["doc001","doc002","doc003","doc004","doc005"]
#IDs must be unique strings -like primary keys in sql
#If you try to add the same ID twice,ChromaDB will raise an error
sample_metadata=[
    {"subject":"Data Engineering","topic":"ETL"},
    {"subject":"Data Engineering","topic":"SQL"},
    {"subject":"Machine Learning","topic":"ML Basics"},
    {"subject":"Data Science","topic":"Python"},
    {"subject":"Artificial Intelligence","topic":"Neural Networks"}
]
#metadata: a list of dictionaries -one dict per document
#Each dict can have any keys you want
#MEtadata is used for filtering later(e.g., only search ML documents)

#Add all documents to the collections
collection.add(
    documents=sample_docs, #The actual text content
    ids=sample_ids,      #unique string IDs
    metadatas=sample_metadata #Extra info aobut each document
)
print(f"Documents added to collections!")
print(f"Total documents now in collection:{collection.count()}")
#Should now show 5

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 70.6MiB/s]


Documents added to collections!
Total documents now in collection:5


In [5]:
query="how do i clean and prepare data?"
results=collection.query(
       query_texts=[query],
       n_results=3
   )
print("result keys available:")
print(list(results.keys()))
print(f'Query:',{query})
print("="*60)
print()
matched_docs=results['documents'][0]
matched_ids=results['ids'][0]
matched_distance=results['distances'][0]
matched_metadata=results['metadatas'][0]
for rank ,(doc,doc_id,dist,meta) in enumerate(zip(matched_docs,matched_ids,matched_distance,matched_metadata)):
     print("Rank",str(rank) +" | "+"ID:",doc_id+" | "+"Distance:"f'{dist:.4f}')
     print("Subject:",meta['subject'])
     print("Topic:",meta['topic'])
     print("Document:",doc)
     print()

result keys available:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']
Query: {'how do i clean and prepare data?'}

Rank 0 | ID: doc004 | Distance:1.1045
Subject: Data Science
Topic: Python
Document: Python Pandas Library is used for data manipulation and cleaning

Rank 1 | ID: doc002 | Distance:1.6139
Subject: Data Engineering
Topic: SQL
Document: SQL select statements retrieve data from database tables

Rank 2 | ID: doc003 | Distance:1.6497
Subject: Machine Learning
Topic: ML Basics
Document: Machine Learning models learn pattern from training data



In [6]:
filtered_query="How do computers learn from examples"
filtered_results=collection.query(
    query_texts=[ filtered_query],
    n_results=3,
    where={"subject":"Machine Learning"}
)
print("Filter  only machine learning documents")
print("="*60)
for rank,(doc,dist,meta) in enumerate(zip(
   filtered_results['documents'][0],
   filtered_results['distances'][0],
   filtered_results['metadatas'][0]),start=1):
  print(f"Rank {rank}|Distance:{dist:.4f}")
  print("Subject:",meta['subject'])
  print("Topic:",meta['topic'])
  print("Document:",doc)
  print()
print("Notice:Only ML documents appear even though ETL and Pandas")

Filter  only machine learning documents
Rank 1|Distance:1.0076
Subject: Machine Learning
Topic: ML Basics
Document: Machine Learning models learn pattern from training data

Notice:Only ML documents appear even though ETL and Pandas


In [7]:
print("DISTANCE TO SIMILARITY CONFESSION")
print("="*50)
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
print("="*50)
distances=[0.85,0.20,0.40,0.65,0.90]
interpretations=["Near identical","Very Similar","Related","Somewhat Related","Not Related"]
for dist,interpretation in zip(distances,interpretations):
  similarity=1-dist
  print(f"{dist:<15.4f}{similarity:<15.4f}{interpretation:<20}")

DISTANCE TO SIMILARITY CONFESSION
Distance       Similarity     Interpretation      
0.8500         0.1500         Near identical      
0.2000         0.8000         Very Similar        
0.4000         0.6000         Related             
0.6500         0.3500         Somewhat Related    
0.9000         0.1000         Not Related         


In [11]:
import pandas as pd
notes_df=pd.read_csv('college_notes.csv')
print("Dataset loaded")
print("Shape:",notes_df.shape)
print("Columns:",list(notes_df.columns))
notes_df.head()

Dataset loaded
Shape: (15, 4)
Columns: ['note_id', 'subject', 'topic', 'content']


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [16]:
print("Notes per subject:")
print(notes_df['subject'].value_counts())
first_note=notes_df.iloc[0]

print("Note ID:",first_note['note_id'])
print("Subject:",first_note['subject'])
print("Title:",first_note['topic'])
print("Note:",first_note['content'])

Notes per subject:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
Note ID: N001
Subject: Data Engineering
Title: ETL Pipelines
Note: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.


In [18]:
all_documents=notes_df['content'].tolist()
all_ids=notes_df['note_id']
all_metadata=[
    {
        "subject":row['subject'],
        "topic":row['topic']
    }
    for _,row in notes_df.iterrows()
]
print("Documents prepared:",len(all_documents))
print("IDs prepared:",len(all_ids))
print("Metadata prepared:",len(all_metadata))
print()
print("Sample ID:",all_ids[0])
print("Sample Subject:",all_metadata[0])
print("Sample Topic:",all_metadata[0])


Documents prepared: 15
IDs prepared: 15
Metadata prepared: 15

Sample ID: N001
Sample Subject: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
Sample Topic: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
